In [38]:
# =======================================================================================
#                       Etape 1 : Préparation et Transformation
# =======================================================================================

In [65]:
# ==================================
# 1. Importation des bibliothèques
# ==================================
from google.cloud import bigquery
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

In [66]:
# =========================================
# 3. Chargement et exploration des données
# =========================================
# Charger le dataset

client = bigquery.Client.from_service_account_json("key.json")

query = """
    SELECT
    *
    FROM `project-a0475dd2-d0c1-4640-aea.Sport_Metrics.mart_physical_condition`

"""

# Executer la requete
query_job = client.query(query)

# Transformer en dataframe
data = query_job.to_dataframe()

# Aperçu des premières lignes
print("Aperçu des données :\n", data.head())

# Informations générales sur les colonnes et les types de données
print("\nInformations sur le dataset :")
print(data.info())

# Statistiques descriptives
print("\nStatistiques descriptives :")
print(data.describe())

C:\Users\fafap\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Aperçu des données :
       Season   player_id                        session_id  Next_Match_ID  \
0  2019-2020  1681114057  FFS_20191020_1681114057_21900010       21900010   
1  2019-2020  1684823847  FFS_20191021_1684823847_21900010       21900010   
2  2019-2020  1681114057  FFS_20191021_1681114057_21900010       21900010   
3  2019-2020  1679015727  FFS_20191021_1679015727_21900010       21900010   
4  2019-2020  1657985451  FFS_20191021_1657985451_21900010       21900010   

      player_name  fi_last_training  rs_last_training  \
0    Abdoulaye Ba             55.11              3.01   
1  Yanis Belkacem             55.33              1.10   
2    Abdoulaye Ba             56.88              2.06   
3  Bryan Fournier             58.30              2.08   
4   Pierre Martin             61.32              0.96   

   recovery_needed_last_training  Fi_before_match  Focus_Level  ...  fg3_pct  \
0                              0             0.00          8.0  ...    0.000   
1           

In [67]:
# ==================================
# 4. Gestion des valeurs manquantes
# ==================================
# Vérification des valeurs manquantes
print(data.isnull().sum())

Season                           0
player_id                        0
session_id                       0
Next_Match_ID                    0
player_name                      0
fi_last_training                 0
rs_last_training                 0
recovery_needed_last_training    0
Fi_before_match                  0
Focus_Level                      0
Strength_Score                   0
Shooting_Accuracy_pct            0
Passing_Accuracy_pct             0
Performance_Score                0
Load_Intensity_Score             0
Injury_Risk                      0
fi_avg_7d                        0
fi_max_7d                        0
training_load_7d                 0
fi_avg_28d                       0
fi_max_28d                       0
training_load_28d                0
Place                            0
Position                         0
Start_position                   0
minutes_played                   0
Points                           0
fg_pct                           0
fg3_pct             

In [68]:
# ==================================================
# 5. Selection et nettoyage des données pertinentes
# ==================================================
# Sélection des données pertinentes
data = data[data["Season"] == '2023-2024']
data["ACWR"] = data["fi_avg_7d"] / data["fi_avg_28d"]

# Aperçu des premières lignes
print("Aperçu des données pertinentes :\n", data.head())

Aperçu des données pertinentes :
          Season   player_id                        session_id  Next_Match_ID  \
5883  2023-2024  1659116774  FFS_20231107_1659116774_22300013       22300013   
5884  2023-2024  1659116774  FFS_20231108_1659116774_22300013       22300013   
5885  2023-2024  1664883672  FFS_20231107_1664883672_22300013       22300013   
5886  2023-2024  1664883672  FFS_20231108_1664883672_22300013       22300013   
5887  2023-2024  1655322242  FFS_20231108_1655322242_22300013       22300013   

          player_name  fi_last_training  rs_last_training  \
5883   Kevin Rousseau             50.69              2.26   
5884   Kevin Rousseau             56.30              1.27   
5885  Zacharie Dupont             59.62              2.33   
5886  Zacharie Dupont             57.66              1.91   
5887  Douglas Hermans             64.36              1.16   

      recovery_needed_last_training  Fi_before_match  Focus_Level  ...  \
5883                              0         

In [69]:
# =======================
# 6. Encodage des données
# =======================
# Encodage des colonnes catégoriques (sans relation d'ordre)

data["Position"] = LabelEncoder().fit_transform(data["Position"])

print(data["Position"].head())

5883    5
5884    5
5885    2
5886    2
5887    1
Name: Position, dtype: int64


In [70]:
# =======================================================================================
#                       Etape 2 : Machine Learning
# =======================================================================================

In [71]:
# Séparation des variables explicatives 'indépendantes' (X) et de la variable cible 'dépendante' (y)
X = data[["Position","fi_last_training","Fi_before_match","ACWR","fi_max_7d","training_load_7d","training_load_28d",
"recovery_needed_last_training","rs_last_training","Focus_Level","minutes_played"]]

y = data["Injury_Risk"]

# Division des données en ensemble d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Dimensions des ensembles :\nEntraînement : {X_train.shape}\nTest : {X_test.shape}")

Dimensions des ensembles :
Entraînement : (1298, 11)
Test : (325, 11)


In [72]:
# ===============================================
# 1. Standardisation des données
# ===============================================

# Standardisation des données
X = pd.DataFrame(StandardScaler().fit_transform(X),columns = X.columns, index = X.index)

In [77]:
# =====================================
# 2. Forêts aléatoires : Classification
# =====================================

# Construction et ajustement de la forêt aléatoire
cf_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
cf_model.fit(X_train, y_train)

# Prédictions sur l'ensemble de test
y_pred = cf_model.predict(X_test)

# Évaluation du modèle
accuracy = accuracy_score(y_test, y_pred)

print(f"Forêt aléatoire :")
print(f'Accuracy : {accuracy:.3f}')
print(classification_report(y_test, y_pred))

Forêt aléatoire :
Accuracy : 0.889
              precision    recall  f1-score   support

         0.0       0.89      1.00      0.94       289
         1.0       0.00      0.00      0.00        36

    accuracy                           0.89       325
   macro avg       0.44      0.50      0.47       325
weighted avg       0.79      0.89      0.84       325



C:\Users\fafap\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\fafap\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\fafap\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [59]:
# ======================
# 3. XGBoost Classifier
# ======================

# poids de la classe positive
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

# Construction et ajustement du modèle
xgb_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
         random_state=42, scale_pos_weight=scale_pos_weight, eval_metric="logloss")

xgb_model.fit(X_train, y_train)

# Prédictions sur l'ensemble de test
y_prob = xgb_model.predict_proba(X_test)[:, 1]

# seuil plus bas pour mieux détecter les blessures
for seuil in [0.20, 0.30, 0.40, 0.5]:
    y_pred = (y_prob >= seuil).astype(int)
    print(f"\nSeuil = {seuil}")
    print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
    print("Precision blessure :", round(precision_score(y_test, y_pred), 3))
    print("Recall blessure :", round(recall_score(y_test, y_pred), 3))


Seuil = 0.2
Accuracy : 0.52
Precision blessure : 0.12
Recall blessure : 0.528

Seuil = 0.3
Accuracy : 0.68
Precision blessure : 0.105
Recall blessure : 0.25

Seuil = 0.4
Accuracy : 0.757
Precision blessure : 0.078
Recall blessure : 0.111

Seuil = 0.5
Accuracy : 0.825
Precision blessure : 0.08
Recall blessure : 0.056


Conclusion:

Puisqu'on arrive pas à prédire les blessures des joueurs car la précision blessure est de 12% max, nous avons opté pour de la prévention en nous focalisant sur le Recall blessures

On constate qu'on a un meilleur Accurancy et Recall (> 0.5) pour un seuil de probabilité de 20%, donc notre analyse sera basée sur ce seuil.

Ce modèle sert plutôt à prévenir qu'à prédire les blessures étant donné qu'on ait pas bcp de joueurs blessés dans les données. Un seuil d'alerte (20%) a été mis en place pour prévenir d'une éventuelle blessure d'un joueur.